# Validación y Métricas de Rendimiento (Secciones 4.2 y 4.3)

Este cuaderno constituye el motor de evaluación de nuestro sistema de detección de carriles. En lugar de procesar las imágenes para visualizarlas, el algoritmo extraerá las variables matemáticas puras de las líneas (pendiente $m$ e intersección $c$) frame a frame.

Con estos datos, calcularemos las métricas propuestas por los autores:
1. **Detección de ambas líneas (Eq. 42):** Ratio de éxito en la detección simultánea del carril izquierdo y derecho.
2. **Pendiente Media (Slope - Eq. 41):** Promedio de la inclinación detectada.
3. **Distancia Media (Eq. 40):** Separación euclidiana (en píxeles) de las líneas a la altura del capó del vehículo.
4. **Consistencia Temporal (Eq. 39):** Porcentaje de fotogramas donde el sistema mantiene las líneas estables sin variaciones bruscas respecto al fotograma anterior.

*(Nota: Corregimos la interpretación matemática del artículo para la métrica de Consistencia, la cual por definición debe ser un ratio entre 0 y 1, evidenciando que el "1.72" reportado en la publicación original es un error tipográfico o de cálculo de los autores).*

In [1]:
# ========================================================
# 1. IMPORTACIONES Y RUTAS DE DATOS
# ========================================================
import cv2
import numpy as np
import math
import pandas as pd
import os

# ATENCIÓN: Cambiar por las rutas locales reales de la BD
RUTA_TUSIMPLE = r"C:\Users\hugot\Downloads\archive\TUSimple\test_set\clips\0531"
RUTA_NOCTURNA = r"C:\Users\hugot\Downloads\BD2\BD2"                            

def cargar_secuencia_masiva(ruta_base, max_frames=2000):
    """
    Recorre todas las subcarpetas de la base de datos, extrae los fotogramas
    y los limita a 'max_frames' para optimizar el tiempo de cómputo.
    """
    fotogramas_totales = []
    
    # Buscamos todas las subcarpetas
    carpetas_videos = [os.path.join(ruta_base, d) for d in os.listdir(ruta_base) 
                       if os.path.isdir(os.path.join(ruta_base, d))]
    
    if not carpetas_videos: 
        print(f"Error: No se encontraron subcarpetas en {ruta_base}.")
        return []
    
    # Extraemos las imágenes de todas las carpetas
    for carpeta in carpetas_videos:
        frames_clip = sorted([os.path.join(carpeta, f) for f in os.listdir(carpeta) 
                              if f.endswith('.jpg') or f.endswith('.png')])
        fotogramas_totales.extend(frames_clip)
        
    # Aplicamos el límite de seguridad (Ej: 2000)
    fotogramas_recortados = fotogramas_totales[:max_frames]
    
    nombre_bd = os.path.basename(ruta_base)
    print(f"Dataset [{nombre_bd}] cargado: {len(fotogramas_recortados)} fotogramas listos para evaluar.")
    
    return fotogramas_recortados

In [2]:
# ========================================================
# 2. PIPELINES MATEMÁTICOS (PHT en ambos modos)
# ========================================================

def obtener_datos_dia(img_bgr):
    """Pipeline Diurno (PHT + Promediado). Devuelve: m_izq, c_izq, m_der, c_der"""
    alto, ancho = img_bgr.shape[:2]
    
    # 1. Preprocesamiento y filtrado de color
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask_amarillo = cv2.inRange(hsv, np.array([15, 80, 80]), np.array([35, 255, 255]))
    mask_blanco = cv2.inRange(hsv, np.array([0, 0, 200]), np.array([180, 30, 255]))
    img_filtrada = cv2.bitwise_and(img_bgr, img_bgr, mask=cv2.bitwise_or(mask_amarillo, mask_blanco))
    img_blur = cv2.GaussianBlur(cv2.cvtColor(img_filtrada, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    img_canny = cv2.Canny(img_blur, 50, 150)
    
    # 2. Máscara ROI (Horizonte 60%)
    poligono = np.array([[(0, alto), (ancho, alto), (int(ancho * 0.75), int(alto * 0.6)), (int(ancho * 0.25), int(alto * 0.6))]])
    mask_roi = np.zeros_like(img_canny)
    cv2.fillPoly(mask_roi, poligono, 255)
    img_roi = cv2.bitwise_and(img_canny, mask_roi)
    
    # 3. Transformada Probabilística (PHT)
    lineas = cv2.HoughLinesP(img_roi, 1, np.pi/180, 20, minLineLength=20, maxLineGap=300)
    p_izq, c_izq, p_der, c_der = [], [], [], []
    
    # 4. Extracción Cartesiana
    if lineas is not None:
        for linea in lineas:
            x1, y1, x2, y2 = linea[0]
            if x1 == x2: continue # Evitar división por cero
            
            slope = (y2 - y1) / (x2 - x1)
            intercept = y1 - slope * x1
            
            # Clasificación (ignorando líneas muy horizontales)
            if slope < -0.5: 
                p_izq.append(slope); c_izq.append(intercept)
            elif slope > 0.5: 
                p_der.append(slope); c_der.append(intercept)

    # 5. Averaging (Promediado de todos los segmentos encontrados)
    if p_izq and p_der:
        return np.mean(p_izq), np.mean(c_izq), np.mean(p_der), np.mean(c_der)
    
    return None, None, None, None


def obtener_datos_noche(img_bgr):
    """Pipeline Nocturno (PHT + Selección por Distancia). Devuelve: m_izq, c_izq, m_der, c_der"""
    alto, ancho = img_bgr.shape[:2]
    
    # 1. Corrección Gamma Adaptativa
    gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    y_average = np.mean(gris)
    gamma_final = 0.7 - (-0.3 / math.log10(max(y_average, 1e-6)))
    tabla = np.array([((i / 255.0) ** (1.0/gamma_final)) * 255 for i in np.arange(0, 256)]).astype("uint8")
    img_gamma = cv2.LUT(img_bgr, tabla)
    
    # 2. Filtrado HSV estricto
    hsv = cv2.cvtColor(img_gamma, cv2.COLOR_BGR2HSV)
    mask_blanco = cv2.inRange(hsv, np.array([0, 0, 180]), np.array([180, 30, 255]))
    mask_amarillo = cv2.inRange(hsv, np.array([15, 80, 80]), np.array([35, 255, 255]))
    mask_total = cv2.bitwise_or(mask_blanco, mask_amarillo)
    
    # 3. Máscara ROI (Horizonte 70%)
    horizonte = 0.70
    poligono = np.array([[(0, alto), (ancho, alto), (int(ancho * 0.75), int(alto * horizonte)), (int(ancho * 0.25), int(alto * horizonte))]])
    mask_roi = np.zeros_like(mask_total)
    cv2.fillPoly(mask_roi, poligono, 255)
    img_roi = cv2.bitwise_and(mask_total, mask_roi)
    
    # 4. Transformada Probabilística (PHT)
    lineas = cv2.HoughLinesP(img_roi, 1, np.pi/180, 20, minLineLength=15, maxLineGap=200)
    cand_izq, cand_der = [], []
    centro_x = ancho / 2
    
    # 5. Extracción y Cálculo de Distancia al centro
    if lineas is not None:
        for linea in lineas:
            x1, y1, x2, y2 = linea[0]
            if x1 == x2: continue
            
            m = (y2 - y1) / (x2 - x1)
            c = y1 - m * x1
            theta = abs(math.atan(m))
            d = abs(((x1 + x2) / 2) - centro_x) # Distancia del punto medio del segmento al centro del coche
            
            # Filtro de ángulos (evita reflejos raros)
            if math.radians(20) < theta < math.radians(80):
                if m < 0: cand_izq.append({'d': d, 'm': m, 'c': c})
                else: cand_der.append({'d': d, 'm': m, 'c': c})

    # 6. Selección (Elegimos solo el segmento más cercano, no promediamos)
    if cand_izq and cand_der:
        mejor_izq = min(cand_izq, key=lambda x: x['d'])
        mejor_der = min(cand_der, key=lambda x: x['d'])
        return mejor_izq['m'], mejor_izq['c'], mejor_der['m'], mejor_der['c']
    
    return None, None, None, None

In [3]:
# ========================================================
# 3. MOTOR DE VALIDACIÓN (Cálculo de métricas por frame)
# ========================================================

def evaluar_rendimiento(rutas_frames, modo_pipeline):
    """
    Recorre las imágenes aplicando el pipeline indicado ("DIA" o "NOCHE")
    y calcula las 4 métricas de rendimiento en tiempo real.
    """
    total_frames = len(rutas_frames)
    if total_frames == 0: return None
    
    frames_detectados = 0
    frames_consistentes = 0
    lista_distancias = []
    lista_pendientes = []
    
    m_izq_ant, m_der_ant = None, None
    
    print(f"\n=> Procesando {total_frames} frames con el algoritmo de {modo_pipeline}...")
    
    for i, ruta in enumerate(rutas_frames):
        img = cv2.imread(ruta)
        if img is None: continue
        alto = img.shape[0]
        
        # 1. Ejecutar el motor matemático correspondiente
        if modo_pipeline == "DIA":
            m_izq, c_izq, m_der, c_der = obtener_datos_dia(img)
        else:
            m_izq, c_izq, m_der, c_der = obtener_datos_noche(img)
            
        # 2. Eq. 42: Detección de ambas líneas
        if m_izq is not None and m_der is not None:
            frames_detectados += 1
            
            # 3. Eq. 41: Slope (Pendiente Media Absoluta)
            slope_medio = (abs(m_izq) + abs(m_der)) / 2.0
            lista_pendientes.append(slope_medio)
            
            # 4. Eq. 40: Distance (Distancia Euclidiana en la base del coche)
            x_base_izq = (alto - c_izq) / m_izq
            x_base_der = (alto - c_der) / m_der
            distancia_px = abs(x_base_der - x_base_izq)
            lista_distancias.append(distancia_px)
            
            # 5. Eq. 39: Consistency (Variación < 15% respecto al frame previo)
            if m_izq_ant is not None and m_der_ant is not None:
                var_izq = abs((m_izq - m_izq_ant) / m_izq_ant)
                var_der = abs((m_der - m_der_ant) / m_der_ant)
                if var_izq < 0.15 and var_der < 0.15:
                    frames_consistentes += 1
            
            # Guardamos estado para el siguiente ciclo
            m_izq_ant, m_der_ant = m_izq, m_der
        else:
            # Si un frame falla, se rompe la cadena de consistencia
            m_izq_ant, m_der_ant = None, None

        # Feedback en consola para que no parezca colgado
        if (i+1) % 500 == 0:
            print(f"     Completados {i+1}/{total_frames} frames...")

    return {
        "Consistency": frames_consistentes / total_frames,
        "Distance (px)": np.mean(lista_distancias) if lista_distancias else 0,
        "Slope": np.mean(lista_pendientes) if lista_pendientes else 0,
        "Detection of Lines": frames_detectados / total_frames
    }

In [4]:
# ========================================================
# 4. EJECUCIÓN PRINCIPAL Y TABLA COMPARATIVA
# ========================================================
from IPython.display import display

# 1. Cargamos hasta 2000 frames de cada base de datos
print("--- 1. CARGANDO DATASETS ---")
frames_dia = cargar_secuencia_masiva(RUTA_TUSIMPLE, max_frames=2000)
frames_noche = cargar_secuencia_masiva(RUTA_NOCTURNA, max_frames=2000)

if frames_dia and frames_noche:
    # 2. Extraemos las métricas reales
    print("\n--- 2. INICIANDO MOTOR DE VALIDACIÓN ---")
    res_dia = evaluar_rendimiento(frames_dia, modo_pipeline="DIA")
    res_noche = evaluar_rendimiento(frames_noche, modo_pipeline="NOCHE")
    
    # 3. Construimos la Tabla formato 'Paper' con Pandas
    df_resultados = pd.DataFrame({
        "Metric": ["Consistency", "Distance (px)", "Slope", "Detection of Lines"],
        "Day Time (SHT)": [
            f"{res_dia['Consistency']:.2f}",
            f"{res_dia['Distance (px)']:.2f}",
            f"{res_dia['Slope']:.2f}",
            f"{res_dia['Detection of Lines']:.2f}"
        ],
        "Night Time (PHT)": [
            f"{res_noche['Consistency']:.2f}",
            f"{res_noche['Distance (px)']:.2f}",
            f"{res_noche['Slope']:.2f}",
            f"{res_noche['Detection of Lines']:.2f}"
        ]
    })
    
    print("\n=======================================================")
    print(" TABLE 3: REAL COMPUTED PERFORMANCE METRICS (N=2000)")
    print("=======================================================")
    display(df_resultados)
else:
    print("\nError fatal: Verifica las rutas de RUTA_TUSIMPLE y RUTA_NOCTURNA en la Celda 2.")

--- 1. CARGANDO DATASETS ---
Dataset [0531] cargado: 2000 fotogramas listos para evaluar.
Dataset [BD2] cargado: 2000 fotogramas listos para evaluar.

--- 2. INICIANDO MOTOR DE VALIDACIÓN ---

=> Procesando 2000 frames con el algoritmo de DIA...
     Completados 500/2000 frames...
     Completados 1000/2000 frames...
     Completados 1500/2000 frames...
     Completados 2000/2000 frames...

=> Procesando 2000 frames con el algoritmo de NOCHE...
     Completados 500/2000 frames...
     Completados 1000/2000 frames...
     Completados 1500/2000 frames...
     Completados 2000/2000 frames...

 TABLE 3: REAL COMPUTED PERFORMANCE METRICS (N=2000)


,Metric,Day Time (SHT),Night Time (PHT)
0,Consistency,0.45,0.51
1,Distance (px),1039.92,1009.56
2,Slope,1.19,0.77
3,Detection of Lines,0.58,0.65
